# C-05 LlamaIndex

이 노트북은 LlamaIndex를 문서 검색과 RAG 후보로 확인하는 예제입니다.

## LlamaIndex가 하는 일

LlamaIndex는 LLM이 참고할 문서를 잘게 관리하고, 검색한 뒤, 검색 결과를 LLM 프롬프트에 연결하는 데 초점을 둔 프레임워크입니다. LangChain이 LLM 애플리케이션 전반의 체인과 도구를 폭넓게 다룬다면, LlamaIndex는 특히 “내 문서에서 찾아 답하기” 흐름에 강합니다.

## RAG란

RAG는 Retrieval-Augmented Generation의 약자입니다. 모델이 기억하고 있는 지식만으로 답하게 하지 않고, 먼저 관련 문서나 이메일을 검색한 뒤 그 내용을 근거로 답하게 만드는 방식입니다. 메일 시스템에서는 과거 메일, 첨부파일, 고객 이력 검색에 유용합니다.

## 이 노트북에서 보는 포인트

- 한글 문서 3개를 `Document`로 만듭니다.
- `OllamaEmbedding`으로 문서를 벡터화합니다.
- `VectorStoreIndex`를 만들어 의미 기반 검색이 가능하게 합니다.
- 질문이 들어오면 관련 문서를 찾고, Ollama LLM이 답변을 생성합니다.

## 언제 적합한가

- 이메일 본문만 분류하는 것을 넘어 과거 메일이나 첨부파일까지 검색해야 할 때
- 고객별 히스토리, 제품 문서, 매뉴얼을 근거로 답변해야 할 때
- 단순 분류보다 “근거 문서 기반 답변”이 중요할 때

이 예제는 인메모리 벡터 인덱스를 사용합니다. 운영 환경에서는 Qdrant, Postgres pgvector 같은 외부 벡터 저장소와 연결하는 구성이 더 일반적입니다.

In [ ]:
try:
    from llama_index.core import Document, VectorStoreIndex
    from llama_index.embeddings.ollama import OllamaEmbedding
    from llama_index.llms.ollama import Ollama
except ImportError as exc:
    raise RuntimeError("이 노트북을 실행하려면 llama-index, llama-index-llms-ollama, llama-index-embeddings-ollama를 설치하세요") from exc

In [ ]:
# 실제 환경에서는 이메일 본문, 첨부파일 텍스트, 제품 문서 등이 Document가 됩니다.
# 여기서는 RAG 흐름을 보기 위해 짧은 한글 문서 3개만 사용합니다.
documents = [
    Document(text="고객이 펌프 예비품 견적과 긴급 납기일 회신을 요청했습니다."),
    Document(text="선박 예비품 구매 발주서 PO-2026-071 접수가 확인되었습니다."),
    Document(text="검사 중 밸브 누수가 확인되어 긴급 서비스 지원 요청이 접수되었습니다."),
]

# llm은 최종 답변 생성을 담당하고, embed_model은 문서 검색용 벡터 생성을 담당합니다.
llm = Ollama(model="llama3.2:latest", request_timeout=120)
embed_model = OllamaEmbedding(model_name="nomic-embed-text")

# VectorStoreIndex는 문서를 임베딩해 검색 가능한 인덱스로 만듭니다.
index = VectorStoreIndex.from_documents(documents, embed_model=embed_model)
query_engine = index.as_query_engine(llm=llm)

In [ ]:
# 질문을 넣으면 LlamaIndex가 관련 문서를 찾고, 그 문서를 바탕으로 LLM이 답합니다.
response = query_engine.query("긴급 서비스 지원과 관련된 이메일은 무엇인가요?")
print(response)